### Эталонная логика инициализации клиента и очистки размышлений модели

In [ ]:
import os
import re
import asyncio
from pyrogram import Client
from ollama import AsyncClient
from dotenv import load_dotenv

# Подгружаем конфигурацию
load_dotenv("../.env")

app = Client(
    name="analyst_userbot",
    api_id=int(os.getenv("API_ID", 0)),
    api_hash=os.getenv("API_HASH"),
    workdir="../src"
)

def clean_think_tags(text: str) -> str:
    return re.sub(re.compile(r'<think>.*?</think>', re.DOTALL), '', text).strip()

print("Компоненты окружения и функции очистки подготовлены.")


### Cквозной сценарий тестирования интеграции сбора истории чата и локального ИИ без вызова блокирующих методов loop.run_until_complete

In [ ]:
async def run_portfolio_analysis():
    # Запуск в контексте уже существующего ядра блокнота
    async with app:
        me = await app.get_me()
        print(f"Сбор истории чата 'Избранное' для аккаунта: {me.first_name}")
        
        messages_text = []
        async Jacks for msg in app.get_chat_history(me.id, limit=5):
            if msg.text:
                author = msg.from_user.first_name if msg.from_user else "Система"
                messages_text.append(f"{author}: {msg.text}")
        
        messages_text.reverse()
        context = "\n".join(messages_text)
        
        print("Отправка контекста в локальную модель deepseek-r1:1.5b...")
        ollama_client = AsyncClient()
        response = await ollama_client.chat(
            model="deepseek-r1:1.5b",
            messages=[
                {"role": "system", "content": "Сделай краткое техническое резюме."},
                {"role": "user", "content": context}
            ]
        )
        
        raw_result = response['message']['content']
        clean_result = clean_think_tags(raw_result)
        
        print("\n--- Результат финального саммари ---")
        print(clean_result)

# Запуск теста напрямую через await
await run_portfolio_analysis()
